# Reducer in LangGraph

## Definition

A **Reducer** in LangGraph is a function that defines **how state values should be combined or updated** when multiple nodes attempt to write to the same state key.

In simple terms, a reducer tells LangGraph:

> **"When multiple values are produced for the same field, how should they be merged into a single value?"**

Without a reducer, LangGraph wouldn't know how to resolve multiple updates to the same state field, especially in **parallel workflows**.

---

# Why Do We Need a Reducer?

Consider a parallel workflow where three nodes execute simultaneously.

```text
                Start
                   │
       ┌───────────┼───────────┐
       ▼           ▼           ▼
    Node A      Node B      Node C
       │           │           │
       └───────────┼───────────┘
                   ▼
                Shared State
```

Suppose each node produces:

```python
Node A
{
    "results": ["Python"]
}
```

```python
Node B
{
    "results": ["LangGraph"]
}
```

```python
Node C
{
    "results": ["AI"]
}
```

Now LangGraph has three updates for the same key:

```python
results
```

Which one should it keep?

- Only Node A?
- Only Node B?
- Only Node C?
- All of them?

A **Reducer** answers this question.

---

# What Does a Reducer Do?

A reducer receives:

- The current value
- The new value

It returns the updated value.

Conceptually:

```python
new_state = reducer(current_value, new_value)
```

For example:

```python
current = ["Python"]

new = ["LangGraph"]

result = reducer(current, new)

# Output
["Python", "LangGraph"]
```

---

# Simple Analogy

Imagine three employees writing notes on the same whiteboard.

Employee A writes:

```text
Python
```

Employee B writes:

```text
LangGraph
```

Employee C writes:

```text
AI
```

Without rules, the notes could overwrite each other.

A reducer acts like the office manager saying:

> "Don't erase previous notes—append the new ones."

Final whiteboard:

```text
Python
LangGraph
AI
```

---

# How Reducers Work

### Without Reducer

```text
Node A → results = ["Python"]

Node B → results = ["AI"]

Final State

results = ["AI"]
```

The second update overwrites the first.

---

### With Reducer

```text
Node A → ["Python"]

Node B → ["AI"]

Reducer

↓

["Python", "AI"]
```

Now both updates are preserved.

---

# Common Reducer Operations

## 1. Append to a List

Combine lists by appending new items.

Example:

```python
["A"]

+

["B"]

↓

["A", "B"]
```

---

## 2. Add Numbers

Sum numeric values.

Example:

```python
10 + 5 = 15
```

---

## 3. Merge Dictionaries

Combine key-value pairs.

Example:

```python
{"name": "Alice"}

+

{"age": 25}

↓

{
    "name": "Alice",
    "age": 25
}
```

---

## 4. Keep Latest Value

Replace the old value with the newest one.

Example:

```python
Old

status = "Running"

New

status = "Completed"

↓

status = "Completed"
```

---

# Real-World Example

Imagine a travel assistant.

Three parallel nodes execute:

```text
Hotel Search

↓

{
  "recommendations": ["Hotel A"]
}
```

```text
Restaurant Search

↓

{
  "recommendations": ["Restaurant B"]
}
```

```text
Attraction Search

↓

{
  "recommendations": ["Museum C"]
}
```

A reducer combines them into:

```python
{
    "recommendations": [
        "Hotel A",
        "Restaurant B",
        "Museum C"
    ]
}
```

---

# Where Are Reducers Most Useful?

Reducers are especially important in:

- Parallel workflows
- Multi-agent systems
- Fan-out/Fan-in architectures
- Concurrent API calls
- Distributed processing
- Aggregating search results
- Collecting outputs from multiple tools

---

# Reducers in LangGraph State

Reducers are typically associated with specific state fields.

For example:

```python
class State(TypedDict):
    messages: Annotated[list, reducer]
```

Whenever multiple nodes update `messages`, LangGraph automatically uses the reducer to combine the values instead of overwriting them.

---

# Benefits of Using Reducers

- Prevent data loss during concurrent updates.
- Safely merge outputs from multiple nodes.
- Simplify state management in parallel workflows.
- Enable scalable multi-agent architectures.
- Make workflows deterministic and predictable.

---

# Things to Consider

- Choose a reducer that matches the type of data (list, number, dictionary, etc.).
- Ensure the reducer is deterministic, producing the same output for the same inputs.
- Avoid reducers with unintended side effects.
- Be mindful of duplicate values when appending lists, if uniqueness is required.

---

# Reducer vs State

| State | Reducer |
|--------|---------|
| Stores workflow data | Defines how state updates are merged |
| Represents the current workflow information | Resolves conflicts when multiple updates occur |
| Shared across nodes | Applied during state updates |
| Contains values | Contains merge logic |

---

# Visualization

```text
                 Shared State
                      │
      ┌───────────────┼───────────────┐
      ▼               ▼               ▼
    Node A         Node B         Node C
      │               │               │
      └───────────────┼───────────────┘
                      ▼
                 Reducer
                      │
                      ▼
               Updated State
```

Example:

```text
Node A

["Python"]
```

```text
Node B

["LangGraph"]
```

```text
Node C

["AI"]
```

```text
Reducer

↓

["Python", "LangGraph", "AI"]
```

---

# Best Practices

- Use reducers whenever multiple nodes update the same state key.
- Match the reducer logic to the data type.
- Keep reducers simple and predictable.
- Test reducers independently before integrating them into workflows.
- Avoid unnecessary complexity in merge logic.

---

# Summary

A **Reducer** in LangGraph is a function that determines how updates to the same state field are combined. It is essential for parallel and concurrent workflows, ensuring that multiple node outputs are merged correctly rather than overwriting each other. By using reducers, LangGraph can safely aggregate results, maintain consistent state, and support scalable workflow patterns such as parallel execution and multi-agent systems.